# 02 · Tools del dominio bancario

**Objetivo:** declarar dos custom tools en Python — `account_status` y `cancel_product` — exponerlas al agente vía un MCP server in-process y disparar la cancelación de la tarjeta `TC-XX1234`.

Las tools simulan el core bancario; en producción ese mismo *contract* lo expone un servicio real. La firma es la misma.

In [ ]:
import asyncio
import nest_asyncio
from typing import Any
from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
)

nest_asyncio.apply()

## 1. `account_status` — solo lectura

Devuelve el estado y los productos asociados a una cuenta. Marcamos `readOnlyHint=True` para que el SDK lo trate como llamada sin efectos secundarios.

In [ ]:
from claude_agent_sdk import ToolAnnotations

@tool(
    name='account_status',
    description='Lista los productos activos de una cuenta de banca digital colombiana.',
    input_schema={'account_id': str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def account_status(args: dict[str, Any]) -> dict[str, Any]:
    return {
        'content': [{
            'type': 'text',
            'text': (
                f"account_id={args['account_id']}; holder=Carlos Auditor; "
                'products=[{id: TC-XX1234, type: credit_card, status: ACTIVE, balance_cop: 2350000}]'
            ),
        }]
    }

## 2. `cancel_product` — escritura con efectos regulatorios

Simula la cancelación. Genera el ticket y deja constancia de los registros normativos (Habeas Data + SFC 029).

In [ ]:
@tool(
    name='cancel_product',
    description=(
        'Cancela un producto financiero (tarjeta, cuenta). Genera ticket de auditoría '
        'y notificación normativa (Ley 1581/2012, SFC 029).'
    ),
    input_schema={
        'account_id': str,
        'product_id': str,
        'product_type': str,
    },
)
async def cancel_product(args: dict[str, Any]) -> dict[str, Any]:
    if args.get('product_type') != 'credit_card':
        return {
            'content': [{'type': 'text', 'text': f"product_type {args.get('product_type')} no soportado"}],
            'is_error': True,
        }
    return {
        'content': [{
            'type': 'text',
            'text': (
                'ticket_id=CCL-2026-0073; state=CANCELLED; '
                f"product_id={args['product_id']}; "
                'regulatory={ley_1581: ok, sfc_029: ok}'
            ),
        }]
    }

## 3. Servidor MCP in-process

`create_sdk_mcp_server` empaqueta las tools y las expone al agente sin necesidad de un proceso aparte.

In [ ]:
banking_server = create_sdk_mcp_server(
    name='banking',
    version='0.1.0',
    tools=[account_status, cancel_product],
)

## 4. Ejecutar el flujo de cancelación

Le decimos al agente lo mínimo: *"cancela la tarjeta de Carlos"*. El SDK orquesta `account_status` → `cancel_product` y nos devuelve el `ResultMessage` final.

In [ ]:
async def run_cancel_flow():
    options = ClaudeAgentOptions(
        model='claude-opus-4-7',
        mcp_servers={'banking': banking_server},
        allowed_tools=[
            'mcp__banking__account_status',
            'mcp__banking__cancel_product',
        ],
        permission_mode='acceptEdits',
        system_prompt=(
            'Eres un asistente bancario auditable. Antes de cancelar un producto '
            'siempre verifica con account_status. Reporta el ticket_id devuelto por la tool. '
            'No afirmes éxito sin haber recibido el ToolResult correspondiente.'
        ),
    )

    user_request = (
        'El cliente Carlos (account_id=ACC-9911) pide cancelar su tarjeta '
        'de crédito TC-XX1234. Procede.'
    )

    async for message in query(prompt=user_request, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if getattr(block, 'type', None) == 'tool_use':
                    print(f"  → tool_use {block.name}({block.input})")
                elif getattr(block, 'text', None):
                    print(f"  texto: {block.text[:120]}{'…' if len(block.text) > 120 else ''}")
        if isinstance(message, ResultMessage):
            print()
            print('=' * 60)
            print('stop_reason :', message.stop_reason)
            print('subtype     :', message.subtype)
            print('result      :', message.result)
            print('cost USD    :', round(message.total_cost_usd or 0, 6))

asyncio.run(run_cancel_flow())

## 5. Lo que ganamos

Confirmamos que el SDK:

1. Acepta tools Python con schema simple.
2. Las invoca en el orden correcto sin que tengamos que escribir el loop a mano.
3. Expone `tool_use` y `tool_result` como bloques diferenciados — cumplen el rol de "señal técnica" sobre la que la UI del prototipo construye el verdict.

El siguiente notebook se enfoca en cómo **leer** esas señales para tomar decisiones deterministas.